# 09 — the offset state, mapped in bias

Session 04, capturing to `protocols/04-offset-state.md`. No light source, cap on,
about three hours, and almost all of it is bias frames.

**What this notebook is for.** Session 03 found that the camera's black level
occupies discrete states about one ADC count apart (`dark_constants.json` →
`offset_state_step` = 0.9931, in 4.1% of 244 frames), and could not say why,
because it varied nothing that might cause it. This notebook varies things:
gain, offset, idle time, and reconfiguration. It captures the three arms and
measures **how the step scales with gain**, which is what separates a post-gain
digital level (H1) from an analog one at the sense node (H2).

**What it is not for.** It pins no term in `sigma^2`, and that is stated first
because CLAUDE.md requires a measurement to name the coefficient it pins down.
What it protects instead is the *validity* of three coefficients already
published — `pedestal`, `R(gain)` and `g(gain)` — every one of which is a
difference against a bias level. If the step is one count everywhere, those
tables are safe. If it scales with gain, it is several counts at 450 and two
published sweeps have a systematic in them.

It is also **not a repair of `D`**: the dark bound is 3,000x below L32's sky
rate, and session 03's 300 s / 600 s confound is deliberately out of scope
here (protocol, *What is out of scope*).

**The predictions are pre-registered and the threshold is not one of them.**
Session 03 detected the state against a fixed 0.5-count threshold, which is
only correct if the step is 0.993 — the thing being tested. `stats.offset_state`
derives its threshold from each peer group instead: half the modal separation,
floored at five times the within-state scatter.

The explaining half is `10`.


## The plan, before the camera is opened

Three hypotheses, separated by how the step scales with gain, using session 01's
own pedestal model `pedestal = A + B * 10**(gain/200)`:

| | mechanism | signature |
|---|---|---|
| **H1** | after the gain stage — a digital or post-ADC level | step **fixed in counts** at every gain |
| **H2** | before it — an analog reference shift, or electrons at the sense node | step **scales with the analog term** `B * 10**(gain/200)`, and jumps at HCG |
| **H3** | a firmware event | distinguished by *when*, not by size: only across a reconfiguration |

The two size predictions differ by a factor of 6 at gain 0 and 10 at gain 450,
against a measurement precision on a plane mean of about 0.007 counts. Every
number below is loaded from `results/`, never retyped.


In [ ]:
import csv
import datetime as dt
import json
import pathlib
import sys
import time

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, fits as F, spatial as SP, stats as ST

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session04"
FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

GAIN, OFFSET = 250, 15                 # the baseline arm: session 03's configuration
ROI = (1408, 568, 1024, 1024)          # not shrunk -- session 03's levels are on this ROI
SETPOINT_C = asi.SETPOINT_C

GAINS = (0, 100, 250, 450)             # every one a swept point in sessions 01 and 02
OFFSETS = (15, 30, 60)                 # 60 is deliberate: see the quarter-unit test
CYCLES, N_PER_SETTING = 3, 20
N_ARM_A = 290
GAPS_S = (2, 15, 60)                   # drawn at random, so idle time is not confounded
N_CSET, N_COPEN = 40, 10               # reconfiguration pairs

SEED = 20260901                        # part of the record: an interleave that cannot
                                       # be reconstructed cannot be checked
FRAME_GAP_S = 0.2
MAX_RETAKES = 3
GATE3_SPREAD = 0.1                     # counts, over arm A's first 20 frames

bias_c = json.loads((RESULTS / "bias_constants.json").read_text())
ptc_c = json.loads((RESULTS / "ptc_constants.json").read_text())
dark_c = json.loads((RESULTS / "dark_constants.json").read_text())

G_E = ptc_c["system_gain"]["value"]                    # e-/count, per gain, session 02
PED_FIT = bias_c["pedestal_fit"]["value"]              # A + B * 10**(gain/200)
PED_PER_OFFSET = bias_c["pedestal_per_offset_unit"]["value"]
HCG = bias_c["hcg_threshold_gain"]["value"]
STEP_ANCHOR = dark_c["offset_state_step"]["value"]     # 0.9931 counts at gain 250
PEDESTAL_PRED = 76.66                                  # bias_sweep.csv, gain 250 offset 15


def analog_term(gain):
    """`B * 10**(gain/200)` -- the part of the pedestal the gain stage amplifies."""
    return PED_FIT["hcg" if gain >= HCG else "lcg"]["B"] * 10 ** (gain / 200)


def predict_h1(gain):
    return STEP_ANCHOR


def predict_h2(gain):
    return STEP_ANCHOR * analog_term(gain) / analog_term(GAIN)


print(f"anchor: {STEP_ANCHOR} counts at gain {GAIN} "
      f"({dark_c['offset_state_step']['notebook']})")
print(f"a quarter of an offset unit is {PED_PER_OFFSET / 4:.4f} counts -- "
      f"{100 * (STEP_ANCHOR / (PED_PER_OFFSET / 4) - 1):+.2f}% from the anchor")
print(f"\n{'gain':>5} {'H1':>8} {'H2':>8} {'ratio':>7} {'g e-/count':>11} "
      f"{'H1 in e-':>9} {'H2 in e-':>9}")
for g in GAINS:
    h1, h2, ge = predict_h1(g), predict_h2(g), G_E[str(g)]
    print(f"{g:5d} {h1:8.3f} {h2:8.3f} {h2 / h1:7.2f} {ge:11.5f} "
          f"{h1 * ge:9.4f} {h2 * ge:9.4f}")


## The schedule, and the seed that reconstructs it

Arm A runs **first and from a cold power-on**: it is the only arm that can see
the onset. B and C follow and may be reordered freely.

Arm B is **interleaved and cycled, never blocked** — session 03's ladder ran
short to long and left an exposure/time confound it could not resolve. Three
cycles is the minimum that lets a setting effect be separated from a time
effect: a state that appears once, mid-run, contaminates one cycle and shows up
as an inconsistency between cycles rather than as a spurious setting effect.

The order is drawn from `SEED`, and the seed is part of the session record.


In [ ]:
def plan_arm_a(rng):
    """290 bias frames at the baseline, each preceded by a gap of 2, 15 or 60 s.

    The gap is drawn rather than fixed so that "time since the last readout" and
    "time since power-on" are estimated from the same stream instead of being
    confounded.  Mean gap ~26 s.
    """
    return [("A", 0, GAIN, OFFSET, None, float(g))
            for g in rng.choice(GAPS_S, size=N_ARM_A)]


def plan_arm_b(rng):
    """Gain and offset, cycled with the order reshuffled each cycle."""
    blocks = []
    for cycle in range(CYCLES):
        for g in rng.permutation(GAINS):
            blocks.append(("Bg", cycle, int(g), OFFSET))
    for cycle in range(CYCLES):
        for o in rng.permutation(OFFSETS):
            blocks.append(("Bo", cycle, GAIN, int(o)))
    return blocks


rng = np.random.default_rng(SEED)
ARM_A = plan_arm_a(rng)
ARM_B = plan_arm_b(rng)

frames = len(ARM_A) + len(ARM_B) * N_PER_SETTING + 2 * (N_CSET + N_COPEN)
gb = frames * ROI[2] * ROI[3] * 2 / 1e9
a_hours = sum(g for *_, g in ARM_A) / 3600

print(f"arm A   {len(ARM_A):4d} frames, mean gap {np.mean([g for *_, g in ARM_A]):.1f} s"
      f"  -> {a_hours:.2f} h of gaps")
print(f"arm B   {len(ARM_B):4d} blocks x {N_PER_SETTING} = "
      f"{len(ARM_B) * N_PER_SETTING} frames, {len(ARM_B)} configuration changes")
print(f"arm C   {N_CSET} set-pairs + {N_COPEN} open-pairs = "
      f"{2 * (N_CSET + N_COPEN)} frames")
print(f"\ntotal   {frames} frames, {gb:.2f} GB  (protocol budget ~830, ~1.7 GB)")
print("\narm B order, cycle by cycle:")
for arm in ("Bg", "Bo"):
    for cycle in range(CYCLES):
        seq = [f"{g}/{o}" for a, c, g, o in ARM_B if a == arm and c == cycle]
        print(f"  {arm} cycle {cycle}: " + "  ".join(seq))


## Gate 1 — white balance, proved in the pixels

Nothing captured before this passes is usable (L01). Reading the control back
only proves the control took; the evidence is a modal step of 16 on all four
planes, on **stored** values, because the test goes vacuous in ADC counts.

It runs again after every gain change in arm B — 12 of them — because the gate
is cheap and a gain change is exactly the kind of reconfiguration that could
re-apply a white balance the camera thinks it still owns.


In [ ]:
existing = list(FRAMES.glob("*.fits"))
if existing:
    print(f"WARNING: {len(existing)} frames already in {FRAMES}.\n"
          "A resumed session breaks arm A's onset question: the second half is\n"
          "no longer 'time since power-on'.  Say so in the record, and treat\n"
          "the two halves as two runs in 10.")


def gate1(mosaic, where):
    """The modal step must be 16 on all four planes, or the session stops."""
    steps = {n: ST.value_step(p) for n, p in SP.split(mosaic).items()}
    bad = {n: s for n, s in steps.items() if s != 16}
    assert not bad, (f"white balance is still being applied at {where}: {bad} -- "
                     "stop the session, nothing captured from here is usable (L01)")
    return steps


rig = asi.open_camera()
print("gain range     ", rig.range("Gain"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
asi.configure(rig, gain=GAIN, offset=OFFSET)
BIAS = rig.min_exposure_s()               # measured, never assumed

for _ in range(2):                        # discards after the configuration change
    asi.capture(rig, BIAS)

for i in range(5):
    mosaic, _ = asi.capture(rig, BIAS)
    print(f"  frame {i}: {gate1(mosaic, 'gate 1')}")
print(f"\ngate 1 passed on five frames.  bias exposure {BIAS * 1e6:.0f} us")


## Gate 2 — the pedestal is where session 01 left it

At gain 250, offset 15, within 5 counts of 76.66. A wide band on purpose: this
catches a wrong *setting*, not a drift, and a wrong setting would make every
level tonight incommensurable with session 03's.


In [ ]:
def levels_of(mosaic):
    """Per-plane means in ADC counts.  One number per plane, and the mean of
    those four is what the state analysis calls a frame's level."""
    return {n: float(ST.to_adc(p).astype(np.float64).mean())
            for n, p in SP.split(mosaic).items()}


mosaic, hdr = asi.capture(rig, BIAS, imagetyp="BIAS")
lv = levels_of(mosaic)
mean_level = float(np.mean(list(lv.values())))

for n, v in lv.items():
    print(f"  {n:2s} {v:8.3f} counts")
print(f"\nmean {mean_level:.3f} vs session 01's {PEDESTAL_PRED} "
      f"({mean_level - PEDESTAL_PRED:+.3f})")
print(f"header says gain {hdr['GAIN']}, offset {hdr['OFFSET']}")
assert abs(mean_level - PEDESTAL_PRED) < 5.0, (
    f"pedestal {mean_level:.2f} is not session 01's {PEDESTAL_PRED} at gain "
    f"{GAIN}/offset {OFFSET} -- check the settings before capturing anything")
assert hdr["GAIN"] == GAIN and hdr["OFFSET"] == OFFSET
print("gate 2 passed")


## Cool down, and log the curve

**Ambient is a variable here, not a footnote.** Arm A's whole claim is about a
machine warming up in a room, so the reading at both ends of the run is part of
the record. Duty is logged per frame for the same reason: session 03's smoke
test showed duty climbing 13 points through the same half hour in which the
pedestal moved 1.1 counts, while sensor temperature sat flat at −10.0.

The protocol asks for the camera to hold in band for a continuous 10 minutes
before arm A's first frame — longer than `asi.SETTLE_S`, because what is being
watched is the *body* equilibrating and not the sensor.


In [ ]:
AMBIENT_START_C = None            # <- record the real reading before running
assert AMBIENT_START_C is not None, "read the room thermometer; it is a variable here"

ARM_A_SETTLE_S = 600.0            # the protocol's 10 continuous minutes in band

cool_path = DATA / "cooldown.csv"
fresh = not cool_path.exists()
cool_log = open(cool_path, "a", newline="")
if fresh:
    cool_log.write("elapsed_s,temp_C,duty_pct\n")


def show(elapsed, temp, duty):
    cool_log.write(f"{elapsed},{temp},{duty}\n")
    cool_log.flush()                     # the reading exists nowhere else
    if int(elapsed) % 30 == 0:
        print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)


try:
    trace = asi.cool_to(rig, SETPOINT_C, settle_s=ARM_A_SETTLE_S, log=show)
finally:
    cool_log.close()

temps = [t for _, t, _ in trace if t is not None]
duties = [d for _, _, d in trace]
print(f"\nsettled in {trace[-1][0]:.0f} s;  {temps[0]} C -> {temps[-1]} C")
print(f"duty {min(duties)}% -> {duties[-1]}% (max {max(duties)}%)")
print("the last 10 minutes of that curve are the body equilibrating, and they "
      "are what arm A is measured against")


## Capture

The library does one frame; the loop is here (`CLAUDE.md`). What the loop
records per frame is the whole apparatus for rule 5's regression: **the gap
before the frame, the cooler duty at the moment it was read, the sensor
temperature and the wall clock**. Three of those are free and the fourth costs
one USB read.

Every frame carries its arm, cycle and gap in its own header as well as in
`frames.csv`, so a frame is self-describing if the CSV is ever lost.


In [ ]:
log_path = DATA / "capture_log.txt"
log_file = open(log_path, "a", encoding="utf8")


def say(msg):
    print(msg, flush=True)
    log_file.write(msg + "\n")
    log_file.flush()            # data/ is gitignored and this is the session record


def in_band(header):
    t = header["CCD-TEMP"]
    return t is not None and abs(t - SETPOINT_C) <= asi.BAND_C


rows = []
t_start = time.monotonic()


def take(arm, block, i, *, gain, offset, cycle=None, gap_s=0.0, event=""):
    """One frame, written and recorded.  Returns `(row, mosaic)`.

    A frame out of band is retaken; one that exhausts the budget is written
    anyway and flagged, because a gap in the series is worse than a frame `10`
    can exclude by its header.  The level is measured here rather than at
    analysis time only so the run can be *watched* -- nothing is decided on it.
    """
    name = f"{arm}{block:02d}_{i:03d}" + (f"_{event}" if event else "")
    path = FRAMES / (name + ".fits")
    if path.exists():
        return None, None                              # resuming; see the warning above
    if gap_s:
        time.sleep(gap_s)
    for attempt in range(MAX_RETAKES + 1):
        mosaic, header = asi.capture(rig, BIAS, imagetyp="BIAS")
        if in_band(header) or attempt == MAX_RETAKES:
            break
    duty = rig.get("CoolPowerPerc")
    header.update(BLOCK=block, ARM=arm, GAP=gap_s, DUTY=duty,
                  CYCLE=-1 if cycle is None else cycle, EVENT=event or "none")
    F.write(path, mosaic, header)
    lv = levels_of(mosaic)
    row = {"arm": arm, "block": block, "i": i, "file": path.name,
           "gain": header["GAIN"], "offset": header["OFFSET"],
           "cycle": -1 if cycle is None else cycle, "event": event or "none",
           "gap_s": gap_s, "duty_pct": duty, "ccd_temp": header["CCD-TEMP"],
           "date_obs": header["DATE-OBS"], "in_band": in_band(header),
           "elapsed_s": time.monotonic() - t_start,
           "level": float(np.mean(list(lv.values()))), **lv}
    rows.append(row)
    time.sleep(FRAME_GAP_S)
    return row, mosaic


def flush_rows():
    """Append the record after every arm, not once at the end.

    Appended rather than rewritten: on a resumed session the earlier rows
    describe frames that cannot be recaptured, and opening this "w" would
    delete the record of half the session while leaving the frames on disk.
    """
    fields = ["arm", "block", "i", "file", "gain", "offset", "cycle", "event",
              "gap_s", "duty_pct", "ccd_temp", "date_obs", "in_band",
              "elapsed_s", "level"] + list(SP.PLANES)
    path = DATA / "frames.csv"
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf8") as fh:
        w = csv.DictWriter(fh, fieldnames=fields)
        if new:
            w.writeheader()
        w.writerows(rows)
    say(f"  wrote {len(rows)} rows to {path.name}")
    rows.clear()


say(f"\n===== session 04 starting {dt.datetime.now():%Y-%m-%d %H:%M} =====")
say(f"seed {SEED}, ambient {AMBIENT_START_C} C, bias exposure {BIAS * 1e6:.0f} us, "
    f"ROI {ROI}")


## Arm A — onset and idle gap

Nothing changes during arm A. No gain change, no offset change, no ROI change,
no reconfiguration: the arm's value is that it is the **null condition**, and
session 03 already showed the state appears in one.

**Gate 3 runs at frame 20.** If the first 20 frames already spread more than
0.1 counts, the camera is switching at minute zero, the onset question is
answered before it is asked, and the analysis must say so rather than pretend to
look for an onset. It is a recorded verdict, not a stop.


In [ ]:
gate3 = None
for i, (arm, block, gain, offset, cycle, gap) in enumerate(ARM_A):
    row, _ = take(arm, block, i, gain=gain, offset=offset, gap_s=gap)
    if i == 19 and len(rows) >= 20:
        first20 = [r["level"] for r in rows[:20]]
        gate3 = float(max(first20) - min(first20))
        say(f"  gate 3: first 20 frames spread {gate3:.4f} counts "
            f"(limit {GATE3_SPREAD})")
        if gate3 > GATE3_SPREAD:
            say("  ^ ALREADY SWITCHING at minute zero.  Arm A cannot see an onset; "
                "10 reports the occupancy and says the onset question was moot")
    if row and i % 10 == 0:
        say(f"  A {i:3d}/{len(ARM_A)}  level {row['level']:.4f}  "
            f"{row['ccd_temp']} C  duty {row['duty_pct']}%  "
            f"[{row['elapsed_s'] / 60:.1f} min]")

levels = [r["level"] for r in rows]
say(f"arm A done: {len(rows)} frames, level {np.mean(levels):.4f} "
    f"+/- {np.std(levels, ddof=1):.4f}, peak-to-peak "
    f"{max(levels) - min(levels):.4f} counts, "
    f"{(time.monotonic() - t_start) / 3600:.2f} h elapsed")
flush_rows()


## Arm B — the decisive test

Twelve gain blocks and nine offset blocks, cycled with the order reshuffled.
Gate 1 runs on the first frame of every block that changed the gain, and two
frames are discarded after each configuration change — the same discipline as
the opening gate, applied 21 more times.

**Offset 60 is here deliberately.** If the step is a quarter of an offset unit,
it is a property of the DAC and unchanged by which offset is selected; if it
scales with the offset *setting*, it is somewhere else entirely.


In [ ]:
current = (GAIN, OFFSET)
for b, (arm, cycle, gain, offset) in enumerate(ARM_B):
    block = 10 + b
    if (gain, offset) != current:
        asi.configure(rig, gain=gain, offset=offset)
        for _ in range(2):
            asi.capture(rig, BIAS)                    # discards after the change
        current = (gain, offset)
    started = time.monotonic()
    steps = None
    for i in range(N_PER_SETTING):
        row, mosaic = take(arm, block, i, gain=gain, offset=offset, cycle=cycle)
        if i == 0 and mosaic is not None:
            steps = gate1(mosaic, f"{arm} block {block}, gain {gain}, offset {offset}")
    lv = [r["level"] for r in rows if r["block"] == block]
    say(f"blk{block:02d} {arm} cycle {cycle}  gain {gain:3d} offset {offset:2d}  "
        f"level {np.mean(lv):9.4f} +/- {np.std(lv, ddof=1):.4f}  "
        f"p-p {max(lv) - min(lv):.4f}  step16 {'ok' if steps else '-'}  "
        f"[{(time.monotonic() - t_start) / 3600:.2f} h]")

asi.configure(rig, gain=GAIN, offset=OFFSET)          # back to the baseline for arm C
for _ in range(2):
    asi.capture(rig, BIAS)
say(f"arm B done, {(time.monotonic() - t_start) / 3600:.2f} h elapsed")
flush_rows()


## Arm C — reconfiguration

**C-set** re-asserts `configure(gain=250, offset=15)` with *identical* values
between two frames: 40 pairs straddling a no-op reconfiguration, against arm A's
290 frames straddling nothing. That is H3's direct test, and it is cheap.

**C-open** closes the camera and reopens it, which drops the TEC (`asi.Rig.close`)
and costs a re-cool. The settle here is `asi.RECOVER_S` and not a full cool-down:
the body stays cold through a fifteen-second gap, so what is being waited out is
the sensor, not the room. Ten pairs is enough to see a large effect, which is
the only kind worth acting on.


In [ ]:
for pair in range(N_CSET):
    take("Cs", 30, pair, gain=GAIN, offset=OFFSET, event="pre")
    asi.configure(rig, gain=GAIN, offset=OFFSET)      # identical values, on purpose
    take("Cs", 30, pair, gain=GAIN, offset=OFFSET, event="post")
say(f"C-set done: {N_CSET} pairs across a no-op reconfiguration")
flush_rows()

for pair in range(N_COPEN):
    take("Co", 40, pair, gain=GAIN, offset=OFFSET, event="pre")
    rig.close()                                       # this drops the cooler
    rig = asi.open_camera()                           # rebind only after the close
    asi.neutralise_white_balance(rig)
    asi.set_roi(rig, *ROI)
    asi.configure(rig, gain=GAIN, offset=OFFSET)
    asi.cool_to(rig, SETPOINT_C, settle_s=asi.RECOVER_S)
    for _ in range(2):
        asi.capture(rig, BIAS)
    mosaic, _ = asi.capture(rig, BIAS)
    gate1(mosaic, f"C-open pair {pair}")              # a reopen is where WB would return
    take("Co", 40, pair, gain=GAIN, offset=OFFSET, event="post")
    say(f"  C-open pair {pair}: reopened and back in band "
        f"[{(time.monotonic() - t_start) / 3600:.2f} h]")
say(f"C-open done: {N_COPEN} pairs across a close/open")
flush_rows()


In [ ]:
AMBIENT_END_C = None      # <- record the real reading, then close the camera

# The protocol's *Record for the session*, written as a file rather than left in
# a variable.  The analysis half is normally run the next morning in a fresh
# kernel, and a seed or an ambient reading that lives only in a dead kernel is
# not a record.  `10` reads this too.
record = {"seed": SEED, "ambient_start_C": AMBIENT_START_C,
          "ambient_end_C": AMBIENT_END_C, "settle_s": ARM_A_SETTLE_S,
          "gate3_spread_counts": gate3, "gate3_limit_counts": GATE3_SPREAD,
          "bias_exposure_s": BIAS, "roi": list(ROI),
          "hours": round((time.monotonic() - t_start) / 3600, 2)}
(DATA / "session_record.json").write_text(json.dumps(record, indent=2))

say(f"ambient {AMBIENT_START_C} C -> {AMBIENT_END_C} C")
say(f"session 04 complete in {record['hours']} h")
rig.close()
log_file.close()
print("camera closed; the TEC is off and the sensor is warming")
print(record)


## The analysis half

The rules below were fixed in `protocols/04-offset-state.md` before the frames
existed, and they are applied in the protocol's own order. Per CFA plane, on
the mosaic, in ADC counts. Never debayer.

| publishes | what |
|---|---|
| `offset_state_frames.csv` | every frame: setting, level, departure, state, duty, temperature, gap |
| `offset_state_settings.csv` | every peer group: step, scatter, threshold, occupancy, rejection cost |
| `offset_state_constants.json` | the verdict, with the losing prediction's residual beside it |

**Entry point.** This half needs the setup cell and nothing else from the
capture half: restart the kernel, run the imports-and-constants cell, then run
from here. Everything the night knew that the frames do not — the seed, the
ambient readings, gate 3's verdict — is in `data/session04/session_record.json`,
written when the camera closed.

The per-frame table is published rather than summarised because rule 3's
questions — run lengths, transition probabilities, whether the process has
memory — are questions about a *sequence*, and a table of block means cannot
answer them.


In [ ]:
import re

import pandas as pd

FRAMES_CSV = RESULTS / "offset_state_frames.csv"
SETTINGS_CSV = RESULTS / "offset_state_settings.csv"
CONSTANTS = RESULTS / "offset_state_constants.json"
NOTEBOOK = "09_offset_state.ipynb"
PLANES = list(SP.PLANES)

RECORD = json.loads((DATA / "session_record.json").read_text())
print("session record:", RECORD)

# One pass over the session.  Per frame, per plane, the mean in ADC counts --
# which is all the level analysis needs.  Read from the frames rather than from
# `frames.csv`: the FITS files are the record, the CSV is the convenience.
recs = []
for path in sorted(FRAMES.glob("*.fits")):
    mosaic, hdr = F.read(path)
    rec = {n: float(ST.to_adc(p).astype(np.float64).mean())
           for n, p in SP.split(mosaic).items()}
    # The frame index lives in the filename and nowhere else; arm C needs it
    # to pair a `pre` with its `post`.
    rec["i"] = int(re.match(r"[A-Za-z]+\d+_(\d+)", path.name).group(1))
    rec.update(file=path.name, arm=hdr["ARM"], block=int(hdr["BLOCK"]),
               cycle=int(hdr["CYCLE"]), event=hdr["EVENT"], gap_s=float(hdr["GAP"]),
               duty_pct=int(hdr["DUTY"]), gain=int(hdr["GAIN"]),
               offset=int(hdr["OFFSET"]), exptime=float(hdr["EXPTIME"]),
               ccd_temp=hdr["CCD-TEMP"], date_obs=hdr["DATE-OBS"],
               roi_w=mosaic.shape[1])
    recs.append(rec)

fr = pd.DataFrame(recs)
fr["t"] = pd.to_datetime(fr.date_obs)
fr = fr.sort_values("t").reset_index(drop=True)
fr["t_min"] = (fr.t - fr.t.min()).dt.total_seconds() / 60
fr["level"] = fr[PLANES].mean(axis=1)
MEASURED_ON = str(fr.t.min().date())

print(f"{len(fr)} frames over {fr.t_min.max():.0f} min, captured {MEASURED_ON}")
print(f"temperature {fr.ccd_temp.min()} to {fr.ccd_temp.max()} C, "
      f"{(fr.ccd_temp.sub(SETPOINT_C).abs() > asi.BAND_C).sum()} frames out of band")
print(f"duty {fr.duty_pct.min()}% to {fr.duty_pct.max()}%")
print(fr.groupby(["arm", "gain", "offset"]).size().to_string())


In [ ]:
# Rule 1.  A peer group is every frame sharing gain, offset, exposure and ROI --
# the only frames whose level a frame has any right to equal.  The threshold is
# derived per group by `stats.offset_state`: half the modal separation, floored
# at 5x the within-state scatter.  Session 03's fixed 0.5-count rule is computed
# alongside, and where the two disagree the modal rule is the published one and
# the disagreement is itself a result (protocol rule 1).
PEER = ["gain", "offset", "exptime", "roi_w"]
SESSION03_THRESHOLD = 0.5

fr["departure"] = np.nan
fr["state"] = 0
fr["far"] = False

setting_rows = []
for key, grp in fr.groupby(PEER):
    st = ST.offset_state(grp.level.values)
    fr.loc[grp.index, "departure"] = st["departure"]
    fr.loc[grp.index, "state"] = st["state"]
    fr.loc[grp.index, "far"] = st["far"]
    fixed = np.abs(st["departure"]) > SESSION03_THRESHOLD
    # Per plane, as a check that the state is a single level shift and not a
    # colour: a mechanism before the CFA would not move four planes together.
    per_plane = {f"step_{n}": (ST.offset_state(grp[n].values)["separation"])
                 for n in PLANES}
    setting_rows.append({
        "gain": key[0], "offset": key[1], "exptime": key[2], "roi_w": key[3],
        "n": len(grp), "level": float(grp.level.mean()),
        "separation": st["separation"], "scatter": st["scatter"],
        "threshold": st["threshold"], "n_states": len(st["centres"]),
        "far": int(st["far"].sum()), "occupancy": float(st["far"].mean()),
        "far_session03_rule": int(fixed.sum()),
        "agrees_with_session03": bool((fixed == st["far"]).all()),
        "worst_steps": st["worst_steps"],
        "resolution_limit": ST.STATE_SPLIT_SIGMAS * st["scatter"],
        **per_plane})

settings = pd.DataFrame(setting_rows).sort_values(["offset", "gain"])
# None means "did not resolve", and it has to survive as NaN rather than as an
# object column: everything downstream does arithmetic on it.
for col in ["separation", "worst_steps"] + [f"step_{n}" for n in PLANES]:
    settings[col] = pd.to_numeric(settings[col], errors="coerce")
cols = ["gain", "offset", "n", "level", "separation", "scatter", "threshold",
        "n_states", "far", "occupancy", "far_session03_rule",
        "agrees_with_session03", "worst_steps"]
print(settings[cols].round(5).to_string(index=False))
print(f"\nlargest departure anywhere: {fr.departure.abs().max():.4f} counts")
print("per-plane separations (a state before the CFA would not move four planes "
      "together):")
print(settings[["gain", "offset"] + [f"step_{n}" for n in PLANES]]
      .round(4).to_string(index=False))


In [ ]:
# Rule 2, the headline: the step against gain, at the fixed offset, against both
# pre-registered predictions.  A group whose states did not resolve contributes
# an upper bound rather than a number -- "no state I can see" is not "no state",
# and the bound is the resolution limit (see `stats.offset_state`).
gain_arm = settings[settings.offset == OFFSET].sort_values("gain")
gain_arm = gain_arm[gain_arm.gain.isin(GAINS)].copy()
gain_arm["h1"] = [predict_h1(g) for g in gain_arm.gain]
gain_arm["h2"] = [predict_h2(g) for g in gain_arm.gain]
gain_arm["g_e_per_count"] = [G_E[str(g)] for g in gain_arm.gain]
gain_arm["step_e"] = gain_arm.separation * gain_arm.g_e_per_count
gain_arm["resid_h1"] = gain_arm.separation - gain_arm.h1
gain_arm["resid_h2"] = gain_arm.separation - gain_arm.h2

resolved = gain_arm[gain_arm.separation.notna()]
print(f"{len(resolved)} of {len(gain_arm)} gains resolved a state\n")

# **A gain with no separation is two different observations and they mean
# opposite things**: either no frame left the base state there (which says
# nothing about the step), or frames did shift by less than the resolution
# limit (which is a real upper bound).  Neither is a zero.  The two are told
# apart by the scatter: a hidden state inflates the group's spread, so an
# unresolved setting whose scatter matches the resolved ones is hiding nothing
# larger than its own limit.
ref_scatter = float(resolved.scatter.median()) if len(resolved) else np.nan
for r in gain_arm[gain_arm.separation.isna()].itertuples():
    excess = float(np.sqrt(max(r.scatter ** 2 - ref_scatter ** 2, 0.0)))
    print(f"  gain {int(r.gain)}: no state resolved.  scatter {r.scatter:.5f} "
          f"against {ref_scatter:.5f} where states did resolve -- excess "
          f"{excess:.5f} counts.  H1 predicted {r.h1:.3f}, H2 {r.h2:.3f}, and "
          f"the limit here is {r.resolution_limit:.4f}")

print(gain_arm[["gain", "separation", "resolution_limit", "h1", "h2",
                "resid_h1", "resid_h2", "step_e", "occupancy"]]
      .round(4).to_string(index=False))


def rms(x):
    return float(np.sqrt(np.mean(np.square(np.asarray(x, float))))) if len(x) else None


RMS_H1, RMS_H2 = rms(resolved.resid_h1), rms(resolved.resid_h2)
if not len(resolved):
    VERDICT = "null"
elif RMS_H1 < RMS_H2 / 2:
    VERDICT = "H1"
elif RMS_H2 < RMS_H1 / 2:
    VERDICT = "H2"
else:
    VERDICT = "undecided"

print(f"\nRMS residual   H1 {RMS_H1}   H2 {RMS_H2}")
print(f"verdict on rule 2: {VERDICT}")

# Two readings that come free, and both are H2 predictions H1 does not make:
# whether the step is constant within a conversion-gain branch, and whether it
# jumps at the HCG threshold.
lcg = resolved[resolved.gain < HCG].separation
hcg = resolved[resolved.gain >= HCG].separation
BRANCH_SPREAD = {"lcg": float(lcg.max() - lcg.min()) if len(lcg) > 1 else None,
                 "hcg": float(hcg.max() - hcg.min()) if len(hcg) > 1 else None}
HCG_JUMP = None
if len(lcg) and len(hcg):
    lo = int(resolved[resolved.gain < HCG].gain.max())
    hi = int(resolved[resolved.gain >= HCG].gain.min())
    HCG_JUMP = {"gains": [lo, hi],
                "measured": float(hcg.min() / lcg.max()),
                "h2_predicts": float(predict_h2(hi) / predict_h2(lo))}
print(f"spread within a branch: {BRANCH_SPREAD}")
print(f"across the branch boundary: {HCG_JUMP}")
print("that is a branch *comparison*, not the jump test: this gain set is "
      f"{GAINS} and the nearest pair straddling {HCG} is 190/200, which arm B "
      "does not shoot.  The step's jump at the threshold is folded into the H2 "
      "prediction already -- it is not separately identifiable here, and rule 2 "
      "is decided on the residuals above")
print(f"\nstep in electrons: " + ", ".join(
    f"gain {int(r.gain)} {r.step_e:.4f} e-" for r in resolved.itertuples()))


In [ ]:
# Rule 2's other half, and the specific mechanism H1 could have: a quarter of an
# offset unit.  If that is what the state is, the step is a property of the
# offset DAC and must not care which offset is selected.  If it scales with the
# offset *setting*, it is somewhere else entirely.
QUARTER = PED_PER_OFFSET / 4
offset_arm = settings[(settings.gain == GAIN)
                      & (settings.offset.isin(OFFSETS))].sort_values("offset")
print(f"a quarter of an offset unit is {QUARTER:.4f} counts\n")
print(offset_arm[["offset", "n", "level", "separation", "scatter", "occupancy"]]
      .round(4).to_string(index=False))

res_off = offset_arm[offset_arm.separation.notna()]
QUARTER_UNIT = None
STEP_VS_OFFSET = None
if len(res_off):
    dev = (res_off.separation / QUARTER - 1).abs().max()
    QUARTER_UNIT = bool(dev < 0.02)
    print(f"\nworst departure from a quarter unit: {100 * dev:.2f}%")
    if len(res_off) > 1:
        STEP_VS_OFFSET = float(np.polyfit(res_off.offset, res_off.separation, 1)[0])
        print(f"step against offset setting: {STEP_VS_OFFSET:+.5f} counts per "
              f"offset unit (a DAC property would give 0)")
    # The pedestal itself is a standing check on the offset arm: 4.0032 counts
    # per offset unit is session 01's, and this arm re-measures it for free.
    per_unit = np.polyfit(offset_arm.offset, offset_arm.level, 1)[0]
    print(f"pedestal per offset unit here: {per_unit:.4f} vs session 01's "
          f"{PED_PER_OFFSET} ({100 * (per_unit / PED_PER_OFFSET - 1):+.2f}%)")


In [ ]:
import itertools

# Rule 3.  Occupancy, transitions, and whether the process has memory -- all of
# it from arm A, because arm A is the only stream where nothing changed.
a = fr[fr.arm == "A"].sort_values("t_min").reset_index(drop=True)
far = a.far.values.astype(int)

WINDOW_MIN = 20
a["window"] = (a.t_min // WINDOW_MIN).astype(int)
occ = a.groupby("window").agg(n=("far", "size"), far=("far", "sum"),
                              occupancy=("far", "mean"),
                              duty=("duty_pct", "mean")).reset_index()
print(f"occupancy by {WINDOW_MIN}-minute window:")
print(occ.round(4).to_string(index=False))

FIRST_FAR = float(a.loc[a.far, "t_min"].min()) if a.far.any() else None
print(f"\nfirst far frame: {FIRST_FAR} min "
      f"(session 03 saw its first at 161 min)")

# Runs, and the transition probability in each direction.
runs = []
for value, group in itertools.groupby(far):
    runs.append((int(value), len(list(group))))
p_enter = (sum(1 for i in range(1, len(far)) if far[i] and not far[i - 1])
           / max((far == 0).sum(), 1))
p_leave = (sum(1 for i in range(1, len(far)) if far[i - 1] and not far[i])
           / max((far == 1).sum(), 1))
far_runs = [n for v, n in runs if v == 1]
near_runs = [n for v, n in runs if v == 0]

print(f"\n{len(far_runs)} far runs, {len(near_runs)} near runs")
print(f"p(near -> far) {p_enter:.5f}   p(far -> near) {p_leave:.5f}")
if far_runs:
    mean_run = float(np.mean(far_runs))
    isolated = float(np.mean([n == 1 for n in far_runs]))
    print(f"far run length: mean {mean_run:.2f}, longest {max(far_runs)}, "
          f"{100 * isolated:.0f}% isolated single frames")
    print(f"  a memoryless process with this mean predicts "
          f"{100 / mean_run:.0f}% isolated")
    MEMORY = "memoryless" if abs(isolated - 1 / mean_run) < 0.15 else "has memory"
    print(f"  reading: {MEMORY} -- a slow drift dithering across a threshold "
          "gives long runs and no isolated frames; an independent per-frame "
          "event gives geometric run lengths")
else:
    mean_run = isolated = None
    MEMORY = "no far frames in arm A"
    print(MEMORY)


In [ ]:
# Rule 5.  Regress occupancy on cooler duty, sensor temperature, elapsed time and
# the randomised gap, in that order of suspicion, and report all four even when
# they are null.  A point-biserial correlation, which is what a 0/1 outcome
# against a continuous regressor is; the band is the null's own scatter.
REGRESSORS = ["duty_pct", "ccd_temp", "t_min", "gap_s"]
band = 2 / np.sqrt(len(a))
regression = {}
print(f"arm A, {len(a)} frames.  |r| under {band:.3f} is consistent with no "
      f"relation at 2 sigma\n")
print(f"{'regressor':>10} {'r':>8} {'mean near':>11} {'mean far':>10} {'verdict':>12}")
for name in REGRESSORS:
    x = a[name].astype(float).values
    if np.std(x) == 0 or not a.far.any():
        regression[name] = None
        print(f"{name:>10} {'--':>8} {x.mean():11.3f} {'--':>10} "
              f"{'no variation' if np.std(x) == 0 else 'no far frames':>12}")
        continue
    r = float(np.corrcoef(x, far)[0, 1])
    regression[name] = r
    print(f"{name:>10} {r:8.4f} {x[far == 0].mean():11.3f} "
          f"{x[far == 1].mean():10.3f} "
          f"{'TRACKS' if abs(r) > band else 'null':>12}")

# The gap is the fourth regressor and it asks a different question from the
# other three: a state that depends on the idle time *before* a frame is a
# sensor-idle effect and has nothing to do with warm-up.
if a.far.any():
    by_gap = a.groupby("gap_s").agg(n=("far", "size"), far=("far", "sum"),
                                    occupancy=("far", "mean")).reset_index()
    print("\noccupancy by idle gap:")
    print(by_gap.round(4).to_string(index=False))


In [ ]:
# H3, tested directly.  Arm C's pairs straddle a reconfiguration; arm A's 290
# frames straddle nothing.  If transitions happen only across a reconfiguration,
# the fix is a protocol rule and the filter becomes a check rather than a gate.
def pair_transitions(arm):
    g = fr[fr.arm == arm]
    if g.empty:
        return None
    wide = g.pivot_table(index="i", columns="event", values="state")
    if not {"pre", "post"}.issubset(wide.columns):
        return None
    changed = (wide["pre"] != wide["post"])
    return {"pairs": int(len(wide)), "changed": int(changed.sum()),
            "rate": float(changed.mean())}


CSET = pair_transitions("Cs")
COPEN = pair_transitions("Co")
BASELINE = float(np.mean([far[i] != far[i - 1] for i in range(1, len(far))]))

print(f"arm A, frame to frame:      {BASELINE:.5f} per frame "
      f"({len(far) - 1} intervals)")
for name, res in (("C-set  (no-op reconfigure)", CSET),
                  ("C-open (close and reopen)", COPEN)):
    if res is None:
        print(f"{name}: no pairs")
        continue
    sd = np.sqrt(max(res["rate"] * (1 - res["rate"]), 1e-9) / res["pairs"])
    print(f"{name}: {res['rate']:.5f} +/- {sd:.5f} "
          f"({res['changed']} of {res['pairs']} pairs)")
print("\nH3 wants a rate across a reconfiguration well above the frame-to-frame "
      "baseline; equality is H3 refuted, whatever rule 2 decides")


In [ ]:
# Rule 6.  The rejection filter costs something, and the cost is measured: what
# fraction of frames a session at each setting would lose to it.  A 4% loss is a
# rounding error; a 40% loss at gain 450 changes how a sweep is budgeted.
cost = settings[["gain", "offset", "n", "occupancy", "separation", "scatter"]].copy()
cost["cost_pct"] = 100 * cost.occupancy
print(cost.round(4).to_string(index=False))
WORST_COST = float(cost.cost_pct.max())
print(f"\nworst rejection cost: {WORST_COST:.1f}% at "
      f"gain {int(cost.loc[cost.cost_pct.idxmax(), 'gain'])}, "
      f"offset {int(cost.loc[cost.cost_pct.idxmax(), 'offset'])}")
print(f"whole session: {100 * fr.far.mean():.1f}% of {len(fr)} frames "
      f"(session 03 measured 4.1% of 244)")

fr.drop(columns=["t"]).round(5).to_csv(FRAMES_CSV, index=False)
settings.round(6).to_csv(SETTINGS_CSV, index=False)
print(f"\nwrote {FRAMES_CSV}")
print(f"wrote {SETTINGS_CSV}")


In [ ]:
N_FRAMES = int(len(fr))


def prov(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": N_FRAMES, "measured_on": MEASURED_ON,
            "notebook": NOTEBOOK, "note": note}


loser = {"H1": ("H2", RMS_H2), "H2": ("H1", RMS_H1)}.get(VERDICT, (None, None))
constants = {
    "offset_state_mechanism": prov(
        VERDICT, "one of H1 (after the gain stage), H2 (before it), "
        "undecided, null", None,
        f"the step measured at gains {sorted(resolved.gain.tolist())} against "
        f"two pre-registered predictions normalised to session 03's "
        f"{STEP_ANCHOR} counts at gain {GAIN}.  RMS residual {RMS_H1} for H1 "
        f"and {RMS_H2} for H2" +
        (f"; the losing prediction {loser[0]} misses by {loser[1]} counts"
         if loser[0] else "") +
        ".  H1 is a digital or post-ADC level and leaves the published sweeps "
        "safe; H2 is analog and puts session 01's R and session 02's g at high "
        "gain in question"),
    "offset_state_step_vs_gain": prov(
        {str(int(r.gain)): (None if np.isnan(r.separation) else round(r.separation, 4))
         for r in gain_arm.itertuples()},
        "ADC counts, per gain",
        {str(int(r.gain)): round(float(r.scatter), 5) for r in gain_arm.itertuples()},
        "measured separation between adjacent black-level states at offset "
        f"{OFFSET}.  **None is not a zero**: it means no state resolved there, "
        "which is either no frame leaving the base state or a step under the "
        "resolution limit, and the uncertainty field carries the scatter that "
        "prices the difference.  H1 predicted " +
        ", ".join(f"{int(r.gain)}: {r.h1:.3f}" for r in gain_arm.itertuples()) +
        "; H2 predicted " +
        ", ".join(f"{int(r.gain)}: {r.h2:.3f}" for r in gain_arm.itertuples())),
    "offset_state_step_electrons": prov(
        {str(int(r.gain)): (None if np.isnan(r.step_e) else round(r.step_e, 4))
         for r in gain_arm.itertuples()},
        "e- per state", None,
        "the same steps through session 02's measured g at each gain.  A step "
        "that is a constant number of *electrons* is a statement about the "
        "sense node and is the most physically specific outcome this session "
        "can reach; a constant number of *counts* is a statement about the "
        "converter or later"),
    "offset_state_within_branch": prov(
        BRANCH_SPREAD, "ADC counts (max minus min separation within a branch)",
        None,
        f"H2 predicts the step varies within a conversion-gain branch and jumps "
        f"at gain {HCG}; H1 predicts neither.  Measured ratio across the "
        f"threshold: {HCG_JUMP}"),
    "offset_state_quarter_offset_unit": prov(
        QUARTER_UNIT, "boolean", None,
        f"whether the step is a quarter of an offset unit ({QUARTER:.4f} counts, "
        f"from session 01's {PED_PER_OFFSET} per unit) and independent of the "
        f"offset selected.  Measured at offsets {sorted(OFFSETS)}; the step "
        f"against the offset setting is {STEP_VS_OFFSET} counts per unit, which "
        "a property of the DAC would leave at zero"),
    "offset_state_occupancy": prov(
        round(float(fr.far.mean()), 5), "fraction of frames", None,
        f"{int(fr.far.sum())} of {N_FRAMES} frames across all three arms, "
        f"against session 03's 4.1% of 244.  Per setting in "
        f"offset_state_settings.csv; worst {WORST_COST:.1f}%.  This is rule 6's "
        "rejection cost: what a session at these settings loses to the filter"),
    "offset_state_transitions": prov(
        {"p_near_to_far": round(float(p_enter), 5),
         "p_far_to_near": round(float(p_leave), 5),
         "mean_far_run": mean_run, "isolated_fraction": isolated,
         "reading": MEMORY},
        "per-frame probability, arm A", None,
        f"arm A only -- {len(a)} frames at a fixed configuration, gaps drawn "
        f"from {GAPS_S} s.  A slow drift dithering across a quantisation "
        "boundary gives long runs and no isolated frames; an independent "
        f"per-frame event gives geometric run lengths.  First far frame at "
        f"{FIRST_FAR} min"),
    "offset_state_reconfiguration": prov(
        {"arm_a_frame_to_frame": round(BASELINE, 5),
         "c_set": CSET, "c_open": COPEN},
        "transition rate per interval", None,
        "H3's direct test: 40 pairs straddling a no-op `configure` with "
        "identical values, and 10 straddling a close/reopen, against arm A's "
        "frame-to-frame baseline.  A rate at the baseline refutes H3 whatever "
        "rule 2 decides; a rate far above it makes the fix a protocol rule -- "
        "do not reconfigure mid-block -- rather than a filter"),
    "offset_state_regressors": prov(
        {k: (None if v is None else round(v, 4)) for k, v in regression.items()},
        "point-biserial r against the far/near label, arm A", round(float(band), 4),
        "rule 5, in its order of suspicion: cooler duty, sensor temperature, "
        "elapsed time, and the idle gap before the frame.  Reported even where "
        "null.  The uncertainty field is the 2-sigma band for no relation.  "
        "Duty is the suspect because session 03 saw it climb 13 points through "
        "the same half hour in which the pedestal moved 1.1 counts, at a flat "
        "-10.0 C; the gap is the one that separates a sensor-idle effect from "
        "a warm-up effect"),
    "offset_state_rule_agreement": prov(
        bool(settings.agrees_with_session03.all()), "boolean", None,
        "whether the modal-separation rule and session 03's fixed 0.5-count "
        "threshold label the same frames.  Where they disagree the modal rule "
        "is the published one and the disagreement is a result (protocol rule "
        "1): a fixed 0.5 is only correct if the step is 0.993 everywhere"),
    "session_record": prov(
        {**RECORD,
         "temp_range_C": [float(fr.ccd_temp.min()), float(fr.ccd_temp.max())],
         "duty_range_pct": [int(fr.duty_pct.min()), int(fr.duty_pct.max())]},
        "the protocol's *Record for the session*", None,
        "the seed is part of the record because an interleave that cannot be "
        "reconstructed cannot be checked.  gate3_spread is the first 20 frames "
        "of arm A against its limit: over it, the camera "
        "was already switching at minute zero and the onset question was moot"),
}

with open(CONSTANTS, "w", encoding="utf8") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS} with {len(constants)} constants, provenance on each")
for k, v in constants.items():
    print(f"  {k:34s} {v['value']}")


## What the session decided

The verdict is `offset_state_mechanism` in `offset_state_constants.json`, and
what each outcome *decides* is the table in `protocols/04-offset-state.md` —
written before the frames existed and not restated here, because a decision
table that exists in two places becomes two decision tables.

The three that matter most:

- **H1** — the published sweeps are safe, the filter becomes a gate in every
  bench protocol, and this closes.
- **H2** — session 01's `R` and session 02's `g` need re-examining at high gain.
  That is a re-analysis of existing frames, not a re-shoot, and it is the next
  session. It also puts session 03's 300 s / 600 s confound back in scope, at
  high gain, where it is answerable in far less than a night.
- **no state in three hours** — reported as such. The 2026-09-01 night stands;
  a null here bounds the recurrence rate and does not delete the finding.

`10` reads these files back and explains them. It measures nothing: if it ever
disagrees with `results/`, `results/` is right and `10` is the bug.
